# DLinear Experiment

ეს notebook აგრძელებს N-BEATS notebook-ის სტრუქტურას, მაგრამ იყენებს **DLinear** მოდელს NeuralForecast-იდან.

Experiment flow:

1. Setup and data loading
2. Common preprocessing from `preprocessing.py`
3. DLinear-specific preprocessing
4. Store-Dept median baseline
5. Train-tail check
6. Validation training + evaluation
7. MLflow/DagsHub logging
8. Hyperparameter sweep for `moving_avg_window`, `input_size`, and training steps

**Important:** ეს DLinear ვერსია არის global univariate forecasting. მოდელი იყენებს მხოლოდ ისტორიულ გაყიდვებს: `unique_id`, `ds`, `y`. Preprocessing-ში შექმნილი სხვა features გამოიყენება სხვა მოდელებისთვის და evaluation-ისთვის, მაგრამ ამ DLinear ექსპერიმენტში მოდელში არ გადადის.

## 1. Setup repo, paths, and packages

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys
import os
import subprocess

REPO_URL = "https://github.com/IrakliZerekidze/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.git"
REPO_DIR = Path("/content/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

print("Working directory:", Path.cwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting


In [2]:
!pip install -q mlflow neuralforecast utilsforecast dagshub

## 2. Download Kaggle data if needed

In [3]:
DATA_DIR = REPO_DIR / "data" / "raw"
PROCESSED_DIR = REPO_DIR / "data" / "processed"

DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

required_files = [
    DATA_DIR / "train.csv.zip",
    DATA_DIR / "test.csv.zip",
    DATA_DIR / "features.csv.zip",
    DATA_DIR / "stores.csv",
]

if not all(p.exists() for p in required_files):
    print("Raw Kaggle files not found. Upload kaggle.json when prompted.")
    from google.colab import files
    files.upload()

    Path.home().joinpath(".kaggle").mkdir(exist_ok=True)
    subprocess.run(["cp", "kaggle.json", str(Path.home() / ".kaggle" / "kaggle.json")], check=True)
    subprocess.run(["chmod", "600", str(Path.home() / ".kaggle" / "kaggle.json")], check=True)

    subprocess.run([
        "kaggle", "competitions", "download",
        "-c", "walmart-recruiting-store-sales-forecasting",
        "-p", str(DATA_DIR)
    ], check=True)

    subprocess.run([
        "unzip", "-o",
        str(DATA_DIR / "walmart-recruiting-store-sales-forecasting.zip"),
        "-d", str(DATA_DIR)
    ], check=True)
else:
    print("Raw Kaggle files already exist. Skipping download.")

Raw Kaggle files already exist. Skipping download.


## 3. Run or load common preprocessing

In [4]:
import importlib
import preprocessing
importlib.reload(preprocessing)

from preprocessing import run_pipeline, load_processed, weighted_mae

processed_files = [
    PROCESSED_DIR / "train_part.parquet",
    PROCESSED_DIR / "valid_part.parquet",
    PROCESSED_DIR / "train_full.parquet",
    PROCESSED_DIR / "test_full.parquet",
]

if all(p.exists() for p in processed_files):
    print("Loading existing processed parquet files...")
    train_part, valid_part, train_full, test_full = load_processed(PROCESSED_DIR)
else:
    print("Running common preprocessing...")
    train_part, valid_part, train_full, test_full = run_pipeline(
        data_dir=DATA_DIR,
        out_dir=PROCESSED_DIR,
        months_valid=3,
        save=True,
    )

print("train_part:", train_part.shape)
print("valid_part:", valid_part.shape)
print("train_full:", train_full.shape)
print("test_full:", test_full.shape)

Loading existing processed parquet files...
train_part: (428409, 33)
valid_part: (41463, 32)
train_full: (476333, 33)
test_full: (115064, 29)


In [5]:
real_train_pairs = set(
    zip(
        train_part[train_part["Weekly_Sales"].notna()]["Store"],
        train_part[train_part["Weekly_Sales"].notna()]["Dept"],
    )
)
valid_pairs = set(zip(valid_part["Store"], valid_part["Dept"]))
valid_only_pairs = valid_pairs - real_train_pairs
train_all_pairs = set(zip(train_part["Store"], train_part["Dept"]))
leaked_pairs_in_train = valid_only_pairs & train_all_pairs

print("Validation-only Store-Dept pairs:", len(valid_only_pairs))
print("Validation-only pairs present in train_part:", len(leaked_pairs_in_train))

Validation-only Store-Dept pairs: 10
Validation-only pairs present in train_part: 0


## 4. Experiment config

In [99]:
MODEL_NAME = "DLinear"

CONFIG = {
    "model": "DLinear",
    "model_type": "global_univariate",

    "target_transform": "none",
    "target_col":  "Weekly_Sales",   # "Weekly_Sales_clipped",
    "evaluation_target_col": "Weekly_Sales",

    "input_size": 112,
    "horizon": 14,
    "freq": "W-FRI",

    "loss": "MAE",
    "learning_rate": 1e-3,
    "batch_size": 128,
    "max_steps": 2500,

    "moving_avg_window": 21,
    "windows_batch_size": 1024,
    "num_lr_decays": 3,
    "scaler_type": "robust",

    "validation_split": "last_3_months",
    "missing_target_strategy": "linear_interpolation",
    "random_seed": 42,

    "experiment_name": "DLinear_Training",
}

RUN_NAME = (
    f"DLinear_{CONFIG['target_transform']}_target"
    f"_input{CONFIG['input_size']}"
    f"_h{CONFIG['horizon']}"
    f"_steps{CONFIG['max_steps']}"
    f"_ma{CONFIG['moving_avg_window']}"
    f"_target_{CONFIG['target_col']}"
)

CONFIG, RUN_NAME

({'model': 'DLinear',
  'model_type': 'global_univariate',
  'target_transform': 'none',
  'target_col': 'Weekly_Sales',
  'evaluation_target_col': 'Weekly_Sales',
  'input_size': 112,
  'horizon': 14,
  'freq': 'W-FRI',
  'loss': 'MAE',
  'learning_rate': 0.001,
  'batch_size': 128,
  'max_steps': 2500,
  'moving_avg_window': 21,
  'windows_batch_size': 1024,
  'num_lr_decays': 3,
  'scaler_type': 'robust',
  'validation_split': 'last_3_months',
  'missing_target_strategy': 'linear_interpolation',
  'random_seed': 42,
  'experiment_name': 'DLinear_Training'},
 'DLinear_none_target_input112_h14_steps2500_ma21_target_Weekly_Sales')

In [81]:
import inspect
from neuralforecast.models import DLinear

print(inspect.signature(DLinear))

(h: int, input_size: int, stat_exog_list=None, hist_exog_list=None, futr_exog_list=None, exclude_insample_y=False, moving_avg_window: int = 25, loss=MAE(), valid_loss=None, max_steps: int = 5000, learning_rate: float = 0.0001, num_lr_decays: int = -1, early_stop_patience_steps: int = -1, val_monitor: str = 'ptl/val_loss', val_check_steps: int = 100, batch_size: int = 32, valid_batch_size: Optional[int] = None, windows_batch_size=1024, inference_windows_batch_size=1024, start_padding_enabled=False, training_data_availability_threshold=0.0, step_size: int = 1, scaler_type: str = 'identity', random_seed: int = 1, drop_last_loader: bool = False, alias: Optional[str] = None, optimizer=None, optimizer_kwargs=None, lr_scheduler=None, lr_scheduler_kwargs=None, dataloader_kwargs=None, **trainer_kwargs)


## 5. DLinear-specific preprocessing helpers

DLinear-ს NeuralForecast-ში იგივე მინიმალური ფორმატი სჭირდება, რაც N-BEATS-ს:

- `unique_id` — Store-Dept time series id
- `ds` — weekly date
- `y` — target value

ამ notebook-ში DLinear არის pure historical-sales მოდელი, ანუ არ ვიყენებთ external covariates-ს.

In [82]:
import numpy as np
import pandas as pd
from typing import Dict, Tuple

def fix_isholiday_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Make IsHoliday stable after merges that may create IsHoliday_x / IsHoliday_y."""
    df = df.copy()

    if "IsHoliday" not in df.columns:
        if "IsHoliday_x" in df.columns:
            df["IsHoliday"] = df["IsHoliday_x"]
        elif "IsHoliday_y" in df.columns:
            df["IsHoliday"] = df["IsHoliday_y"]
        else:
            raise KeyError("No IsHoliday column found.")

    drop_cols = [c for c in ["IsHoliday_x", "IsHoliday_y"] if c in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)

    df["IsHoliday"] = df["IsHoliday"].astype(bool)
    return df

def fill_missing_targets_for_series(df: pd.DataFrame, target_col: str, strategy: str) -> pd.DataFrame:
    df = df.copy()
    df = df.sort_values(["Store", "Dept", "Date"])

    if strategy == "zero":
        df[target_col] = df[target_col].fillna(0)
    elif strategy == "linear_interpolation":
        df[target_col] = (
            df.groupby(["Store", "Dept"])[target_col]
            .transform(lambda s: s.interpolate(method="linear", limit_area="inside"))
        )
        df[target_col] = df[target_col].fillna(0)
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    return df

def make_neuralforecast_df(
    df: pd.DataFrame,
    target_col: str = "Weekly_Sales_clipped",
    target_transform: str = "none",
) -> pd.DataFrame:
    """Convert Walmart dataframe to NeuralForecast format: unique_id, ds, y."""
    df = df.copy()
    df["unique_id"] = df["Store"].astype(str) + "_" + df["Dept"].astype(str)
    df["ds"] = pd.to_datetime(df["Date"])
    df["was_missing_week"] = df["Weekly_Sales"].isna().astype(int)

    df = fill_missing_targets_for_series(
        df,
        target_col=target_col,
        strategy=CONFIG["missing_target_strategy"],
    )

    if target_transform == "log1p":
        df["y"] = np.log1p(df[target_col].clip(lower=0))
    elif target_transform == "none":
        df["y"] = df[target_col]
    else:
        raise ValueError(f"Unknown target_transform: {target_transform}")

    return df[["unique_id", "ds", "y", "IsHoliday"]].copy()

def inverse_target_transform(preds: pd.DataFrame, model_col: str, target_transform: str) -> pd.DataFrame:
    """Convert model predictions back to original sales scale."""
    preds = preds.rename(columns={model_col: "pred_transformed"}).copy()

    if target_transform == "log1p":
        preds["pred"] = np.expm1(preds["pred_transformed"])
    elif target_transform == "none":
        preds["pred"] = preds["pred_transformed"]
    else:
        raise ValueError(f"Unknown target_transform: {target_transform}")

    preds["pred"] = preds["pred"].clip(lower=0)
    return preds

def evaluate_wmae(eval_df: pd.DataFrame) -> Dict[str, float]:
    """Compute competition WMAE plus holiday/non-holiday breakdown."""
    eval_target_col = CONFIG["evaluation_target_col"]

    y_true = eval_df[eval_target_col].fillna(0).values
    y_pred = eval_df["pred"].fillna(0).values
    is_holiday = eval_df["IsHoliday"].values

    metrics = {"wmae": float(weighted_mae(y_true, y_pred, is_holiday))}

    holiday_mask = eval_df["IsHoliday"] == True
    nonholiday_mask = eval_df["IsHoliday"] == False

    if holiday_mask.any():
        metrics["holiday_wmae"] = float(weighted_mae(
            eval_df.loc[holiday_mask, eval_target_col].fillna(0).values,
            eval_df.loc[holiday_mask, "pred"].fillna(0).values,
            eval_df.loc[holiday_mask, "IsHoliday"].values,
        ))
    else:
        metrics["holiday_wmae"] = np.nan

    metrics["nonholiday_wmae"] = float(weighted_mae(
        eval_df.loc[nonholiday_mask, eval_target_col].fillna(0).values,
        eval_df.loc[nonholiday_mask, "pred"].fillna(0).values,
        eval_df.loc[nonholiday_mask, "IsHoliday"].values,
    ))

    return metrics

for name in ["train_part", "valid_part", "train_full", "test_full"]:
    globals()[name] = fix_isholiday_columns(globals()[name])

print("IsHoliday columns fixed.")
print("Validation horizon:", CONFIG["horizon"])
print("Number of Store-Dept series:", train_part[["Store", "Dept"]].drop_duplicates().shape[0])

IsHoliday columns fixed.
Validation horizon: 14
Number of Store-Dept series: 3321


## 6. Baseline: Store-Dept median

In [83]:
def store_dept_median_baseline(train_df: pd.DataFrame, actual_df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, float]]:
    target_col = CONFIG["target_col"]

    baseline = (
        train_df
        .groupby(["Store", "Dept"])[target_col]
        .median()
        .reset_index()
        .rename(columns={target_col: "pred"})
    )

    eval_df = actual_df.merge(baseline, on=["Store", "Dept"], how="left")
    eval_df["pred"] = eval_df["pred"].fillna(train_df[target_col].median())
    metrics = evaluate_wmae(eval_df)
    return eval_df, metrics

baseline_valid_eval, baseline_valid_metrics = store_dept_median_baseline(train_part, valid_part)

print(f"Baseline valid WMAE: {baseline_valid_metrics['wmae']:.4f}")
print(f"Baseline holiday WMAE: {baseline_valid_metrics['holiday_wmae']:.4f}")
print(f"Baseline non-holiday WMAE: {baseline_valid_metrics['nonholiday_wmae']:.4f}")

Baseline valid WMAE: 2245.1593
Baseline holiday WMAE: 2518.7795
Baseline non-holiday WMAE: 2139.7540


## 7. Train/evaluate helper

DLinear-ს ორჯერ ვუშვებთ:

- **train-tail check** — train-ის ბოლო 14 კვირას გამოვყოფთ და ვამოწმებთ, როგორ generalize-დება მოდელი training period-ის შიგნით;
- **validation** — მოდელი სწავლობს `train_part`-ზე და ფასდება ბოლო 3 თვეზე, ანუ `valid_part`-ზე.

In [84]:
from neuralforecast import NeuralForecast
from neuralforecast.models import DLinear
from neuralforecast.losses.pytorch import MAE

def build_dlinear(config: Dict) -> NeuralForecast:
    model_kwargs = dict(
        h=config["horizon"],
        input_size=config["input_size"],
        moving_avg_window=config["moving_avg_window"],
        loss=MAE(),
        max_steps=config["max_steps"],
        learning_rate=config["learning_rate"],
        batch_size=config["batch_size"],
        random_seed=config["random_seed"],
        windows_batch_size=config["windows_batch_size"],
        num_lr_decays=config["num_lr_decays"],
    )

    if config.get("scaler_type") is not None:
        model_kwargs["scaler_type"] = config["scaler_type"]

    print(model_kwargs)
    model = DLinear(**model_kwargs)

    return NeuralForecast(models=[model], freq=config["freq"])

def fit_predict_evaluate(
    train_df_raw: pd.DataFrame,
    actual_df_raw: pd.DataFrame,
    config: Dict,
    label: str,
):
    train_nf = make_neuralforecast_df(
        train_df_raw,
        target_col=config["target_col"],
        target_transform=config["target_transform"],
    )

    print("Training dataframe columns:", train_nf.columns.tolist())
    print("Training date range:", train_nf["ds"].min(), "→", train_nf["ds"].max())
    print("Training series:", train_nf["unique_id"].nunique())

    nf = build_dlinear(config)
    nf.fit(df=train_nf[["unique_id", "ds", "y"]])

    preds = nf.predict()
    preds = inverse_target_transform(
        preds,
        model_col=config["model"],
        target_transform=config["target_transform"],
    )

    actual = actual_df_raw.copy()
    actual["unique_id"] = actual["Store"].astype(str) + "_" + actual["Dept"].astype(str)
    actual["ds"] = pd.to_datetime(actual["Date"])

    eval_df = actual.merge(
        preds[["unique_id", "ds", "pred", "pred_transformed"]],
        on=["unique_id", "ds"],
        how="left",
    )

    eval_df["pred"] = eval_df["pred"].fillna(0).clip(lower=0)
    metrics = evaluate_wmae(eval_df)

    print(f"\n{label} metrics")
    print(f"WMAE: {metrics['wmae']:.4f}")
    print(f"Holiday WMAE: {metrics['holiday_wmae']:.4f}")
    print(f"Non-holiday WMAE: {metrics['nonholiday_wmae']:.4f}")

    return nf, preds, eval_df, metrics

## 8. Train-tail check

In [100]:
all_train_dates = sorted(train_part["Date"].unique())
tail_dates = all_train_dates[-CONFIG["horizon"]:]

train_inner = train_part[~train_part["Date"].isin(tail_dates)].copy()
train_tail = train_part[train_part["Date"].isin(tail_dates)].copy()

print("train_inner date range:", train_inner["Date"].min(), "→", train_inner["Date"].max())
print("train_tail date range:", train_tail["Date"].min(), "→", train_tail["Date"].max())
print("train_tail weeks:", train_tail["Date"].nunique())

baseline_train_tail_eval, baseline_train_tail_metrics = store_dept_median_baseline(train_inner, train_tail)
print(f"Baseline train-tail WMAE: {baseline_train_tail_metrics['wmae']:.4f}")

nf_train_tail, train_tail_preds, train_tail_eval, train_tail_metrics = fit_predict_evaluate(
    train_inner,
    train_tail,
    CONFIG,
    label="DLinear train-tail",
)

train_inner date range: 2010-02-05 00:00:00 → 2012-04-13 00:00:00
train_tail date range: 2012-04-20 00:00:00 → 2012-07-20 00:00:00
train_tail weeks: 14
Baseline train-tail WMAE: 2201.0359


INFO:lightning_fabric.utilities.seed:Seed set to 42


Training dataframe columns: ['unique_id', 'ds', 'y', 'IsHoliday']
Training date range: 2010-02-05 00:00:00 → 2012-04-13 00:00:00
Training series: 3321
{'h': 14, 'input_size': 112, 'moving_avg_window': 21, 'loss': MAE(), 'max_steps': 2500, 'learning_rate': 0.001, 'batch_size': 128, 'random_seed': 42, 'windows_batch_size': 1024, 'num_lr_decays': 3, 'scaler_type': 'robust'}


INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.callbacks.model_summary:
  | Name          | Type          | Params | Mode 
--------------------------------------------------------
0 | loss          | MAE           | 0      | train
1 | padder_train  | ConstantPad1d | 0      | train
2 | scaler        | TemporalNorm  | 0      | train
3 | decomp        | SeriesDecomp  | 0      | train
4 | linear_trend  | Linear        | 1.6 K  | train
5 | linear_season | Linear        | 1.6 K  | train
--------------------------------------------------------
3.2 K     Trainable params
0         Non-trainable params
3.2 K     Total params
0.013     Total estimated model params size (MB)
8         Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=2500` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]


DLinear train-tail metrics
WMAE: 2447.3917
Holiday WMAE: nan
Non-holiday WMAE: 2447.3917


## 9. Validation training and evaluation

In [101]:
nf_valid, valid_preds, valid_eval, valid_metrics = fit_predict_evaluate(
    train_part,
    valid_part,
    CONFIG,
    label="DLinear validation",
)

improvement = baseline_valid_metrics["wmae"] - valid_metrics["wmae"]
improvement_pct = 100 * improvement / baseline_valid_metrics["wmae"]
generalization_gap = valid_metrics["wmae"] - train_tail_metrics["wmae"]

summary = pd.DataFrame([
    {
        "split": "train_tail",
        "baseline_wmae": baseline_train_tail_metrics["wmae"],
        "model_wmae": train_tail_metrics["wmae"],
        "holiday_wmae": train_tail_metrics["holiday_wmae"],
        "nonholiday_wmae": train_tail_metrics["nonholiday_wmae"],
    },
    {
        "split": "valid",
        "baseline_wmae": baseline_valid_metrics["wmae"],
        "model_wmae": valid_metrics["wmae"],
        "holiday_wmae": valid_metrics["holiday_wmae"],
        "nonholiday_wmae": valid_metrics["nonholiday_wmae"],
    },
])

display(summary)

print(f"\nValidation improvement over baseline: {improvement:.4f} ({improvement_pct:.2f}%)")
print(f"Generalization gap valid - train_tail: {generalization_gap:.4f}")

if generalization_gap > 300:
    print("⚠️ Validation is much worse than train-tail. Possible overfitting or distribution shift.")
elif valid_metrics["wmae"] > baseline_valid_metrics["wmae"]:
    print("❌ Model is worse than baseline. This setup is probably underperforming.")
else:
    print("✅ Model is better than baseline and train-tail/valid gap is not extreme.")

INFO:lightning_fabric.utilities.seed:Seed set to 42


Training dataframe columns: ['unique_id', 'ds', 'y', 'IsHoliday']
Training date range: 2010-02-05 00:00:00 → 2012-07-20 00:00:00
Training series: 3321
{'h': 14, 'input_size': 112, 'moving_avg_window': 21, 'loss': MAE(), 'max_steps': 2500, 'learning_rate': 0.001, 'batch_size': 128, 'random_seed': 42, 'windows_batch_size': 1024, 'num_lr_decays': 3, 'scaler_type': 'robust'}


INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.callbacks.model_summary:
  | Name          | Type          | Params | Mode 
--------------------------------------------------------
0 | loss          | MAE           | 0      | train
1 | padder_train  | ConstantPad1d | 0      | train
2 | scaler        | TemporalNorm  | 0      | train
3 | decomp        | SeriesDecomp  | 0      | train
4 | linear_trend  | Linear        | 1.6 K  | train
5 | linear_season | Linear        | 1.6 K  | train
--------------------------------------------------------
3.2 K     Trainable params
0         Non-trainable params
3.2 K     Total params
0.013     Total estimated model params size (MB)
8         Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=2500` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]


DLinear validation metrics
WMAE: 1342.3069
Holiday WMAE: 1359.7448
Non-holiday WMAE: 1335.5893


,split,baseline_wmae,model_wmae,holiday_wmae,nonholiday_wmae
0,train_tail,2201.035939,2447.391660,NaN,2447.391660
1,valid,2245.159307,1342.306853,1359.74476,1335.589339



Validation improvement over baseline: 902.8525 (40.21%)
Generalization gap valid - train_tail: -1105.0848
✅ Model is better than baseline and train-tail/valid gap is not extreme.


## 10. MLflow / DagsHub setup

In [102]:
import dagshub
import mlflow

dagshub.init(
    repo_owner="izere23",
    repo_name="ML-Final-Walmart-Recruiting-Store-Sales-Forecasting",
    mlflow=True,
)

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("DLinear_Training")

print("MLflow tracking URI:", mlflow.get_tracking_uri())

Initialized MLflow to track repo "izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting"

Repository izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting initialized!

MLflow tracking URI: https://dagshub.com/izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.mlflow


## 11. Save artifacts and log run

In [103]:
import json
from pathlib import Path

EXPERIMENT_NAME = "DLinear_Training"
ARTIFACT_DIR = Path("artifacts") / RUN_NAME
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

valid_preds.to_csv(ARTIFACT_DIR / "valid_predictions.csv", index=False)
valid_eval.to_csv(ARTIFACT_DIR / "valid_eval.csv", index=False)
train_tail_eval.to_csv(ARTIFACT_DIR / "train_tail_eval.csv", index=False)
summary.to_csv(ARTIFACT_DIR / "metrics_summary.csv", index=False)

feature_summary = {
    "used_by_dlinear_v1": ["unique_id", "ds", "y"],
    "used_for_evaluation": ["IsHoliday"],
    "created_by_common_preprocessing_but_not_used_by_dlinear_v1": [
        "Temperature", "Fuel_Price", "MarkDown1", "MarkDown2", "MarkDown3",
        "MarkDown4", "MarkDown5", "MarkDown1_exists", "MarkDown2_exists",
        "MarkDown3_exists", "MarkDown4_exists", "MarkDown5_exists",
        "CPI", "Unemployment", "Type", "Size", "Year", "Month",
        "WeekOfYear", "Week_sin", "Week_cos", "is_superbowl",
        "is_labor_day", "is_thanksgiving", "is_christmas",
    ],
}

with open(ARTIFACT_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=4)

with open(ARTIFACT_DIR / "feature_summary.json", "w") as f:
    json.dump(feature_summary, f, indent=4)

MODEL_DIR = ARTIFACT_DIR / "model"

if mlflow.active_run():
    mlflow.end_run()

mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME):
    mlflow.log_params(CONFIG)

    mlflow.log_metric("baseline_train_tail_wmae", baseline_train_tail_metrics["wmae"])
    mlflow.log_metric("train_tail_wmae", train_tail_metrics["wmae"])
    mlflow.log_metric("train_tail_holiday_wmae", train_tail_metrics["holiday_wmae"])
    mlflow.log_metric("train_tail_nonholiday_wmae", train_tail_metrics["nonholiday_wmae"])

    mlflow.log_metric("baseline_store_dept_median_wmae", baseline_valid_metrics["wmae"])
    mlflow.log_metric("valid_wmae", valid_metrics["wmae"])
    mlflow.log_metric("holiday_wmae", valid_metrics["holiday_wmae"])
    mlflow.log_metric("nonholiday_wmae", valid_metrics["nonholiday_wmae"])
    mlflow.log_metric("improvement_over_baseline", improvement)
    mlflow.log_metric("improvement_over_baseline_pct", improvement_pct)
    mlflow.log_metric("generalization_gap_valid_minus_train_tail", generalization_gap)

    mlflow.log_artifacts(str(ARTIFACT_DIR), artifact_path="artifacts")

    nf_valid.save(path=str(MODEL_DIR), overwrite=True)
    mlflow.log_artifacts(str(MODEL_DIR), artifact_path="model")

print("✅ DLinear metrics, parameters, artifacts, and model were logged successfully.")

🏃 View run DLinear_none_target_input112_h14_steps2500_ma21_target_Weekly_Sales at: https://dagshub.com/izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.mlflow/#/experiments/2/runs/2ef9bf49352a4c038ba42c4d04cbf1f0
🧪 View experiment at: https://dagshub.com/izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.mlflow/#/experiments/2
✅ DLinear metrics, parameters, artifacts, and model were logged successfully.


## 12. DLinear sweep

DLinear-ში ყველაზე მნიშვნელოვანი ექსპერიმენტები:

- `moving_avg_window` — რა სიგრძის moving average-ით დაიშალოს სერია trend/residual ნაწილებად;
- `input_size` — რამდენი წარსული კვირა დაინახოს მოდელმა;
- `max_steps` — რამდენად დიდხანს იტრენინგოს.

Walmart-ის მონაცემებში 52 და 104 კვირიანი input ლოგიკურია, რადგან ამოცანას აქვს წლიური სეზონურობა.

In [ ]:
sweep_experiments = [
    {"name": "ma13_input52_steps1000", "moving_avg_window": 13, "input_size": 52, "max_steps": 1000},
    {"name": "ma25_input52_steps1000", "moving_avg_window": 25, "input_size": 52, "max_steps": 1000},
    {"name": "ma25_input104_steps1500", "moving_avg_window": 25, "input_size": 104, "max_steps": 1500},
    {"name": "ma51_input104_steps1500", "moving_avg_window": 51, "input_size": 104, "max_steps": 1500},
    {"name": "ma25_input104_steps2000", "moving_avg_window": 25, "input_size": 104, "max_steps": 2000},
]

sweep_results = []

for exp in sweep_experiments:
    exp_config = CONFIG.copy()
    exp_config.update({k: v for k, v in exp.items() if k != "name"})
    exp_run_name = f"DLinear_sweep_{exp['name']}"

    nf_exp, preds_exp, eval_exp, metrics_exp = fit_predict_evaluate(
        train_part,
        valid_part,
        exp_config,
        label=exp_run_name,
    )

    improvement_exp = baseline_valid_metrics["wmae"] - metrics_exp["wmae"]
    improvement_pct_exp = 100 * improvement_exp / baseline_valid_metrics["wmae"]

    if mlflow.active_run():
        mlflow.end_run()

    mlflow.set_experiment(EXPERIMENT_NAME)

    with mlflow.start_run(run_name=exp_run_name):
        mlflow.log_params(exp_config)
        mlflow.log_metric("valid_wmae", metrics_exp["wmae"])
        mlflow.log_metric("holiday_wmae", metrics_exp["holiday_wmae"])
        mlflow.log_metric("nonholiday_wmae", metrics_exp["nonholiday_wmae"])
        mlflow.log_metric("baseline_store_dept_median_wmae", baseline_valid_metrics["wmae"])
        mlflow.log_metric("improvement_over_baseline", improvement_exp)
        mlflow.log_metric("improvement_over_baseline_pct", improvement_pct_exp)

    sweep_results.append({
        "name": exp["name"],
        "moving_avg_window": exp_config["moving_avg_window"],
        "input_size": exp_config["input_size"],
        "max_steps": exp_config["max_steps"],
        "valid_wmae": metrics_exp["wmae"],
        "holiday_wmae": metrics_exp["holiday_wmae"],
        "nonholiday_wmae": metrics_exp["nonholiday_wmae"],
        "improvement_pct": improvement_pct_exp,
    })

sweep_df = pd.DataFrame(sweep_results).sort_values("valid_wmae")
display(sweep_df)

sweep_df.to_csv(Path("artifacts") / "dlinear_sweep_results.csv", index=False)